In [ ]:
# Top-level constants (edit here)
from pathlib import Path

NUM_FEATURES = 100  # change this number as required
OUTPUT_DIR = Path('../data/generated')
OUTPUT_FN = f'points_{NUM_FEATURES}.parquet'

# UK approximate lon/lat bounding box (W, S, E, N) in EPSG:4326
UK_BBOX = (-8.65, 49.9, 1.76, 60.84)  # (min_lon, min_lat, max_lon, max_lat)
CRS = 'EPSG:4326'  # same as GeoJSON / deck.gl expectations

print('NUM_FEATURES =', NUM_FEATURES)

NUM_FEATURES = 100


In [ ]:
# Generate a deterministic grid of lon/lat points inside the UK bbox
import math
import numpy as np
from shapely.geometry import Point

min_lon, min_lat, max_lon, max_lat = UK_BBOX
# Choose grid dimensions so that nx * ny >= NUM_FEATURES and roughly square
nx = int(math.sqrt(NUM_FEATURES))
if nx < 1:
    nx = 1
ny = math.ceil(NUM_FEATURES / nx)

lons = np.linspace(min_lon, max_lon, nx)
lats = np.linspace(min_lat, max_lat, ny)

fids = []
names = []
values = []  # floats
geoms = []

count = 0
for j, lat in enumerate(lats):
    for i, lon in enumerate(lons):
        if count >= NUM_FEATURES:
            break
        fids.append(int(count))
        names.append(f'pt_{count}')
        # value as a float (example: sequential float, you can change to any formula)
        values.append(float(count) + 0.5)
        geoms.append(Point(float(lon), float(lat)))
        count += 1
    if count >= NUM_FEATURES:
        break

print(f'Generated {len(fids)} features (grid {nx}x{ny})')

Generated 100 features (grid 10x10)


In [ ]:
# Build GeoDataFrame
import geopandas as gpd

print('geopandas', gpd.__version__)
gdf = gpd.GeoDataFrame(
    {
        'fid': fids,
        'name': names,
        'value': values,
    },
    geometry=geoms,
    crs=CRS,
)

# show summary
display(gdf.head())
print(gdf.dtypes)

geopandas 1.1.1


,fid,name,value,geometry
0,0,pt_0,0.5,POINT (-8.65 49.9)
1,1,pt_1,1.5,POINT (-7.49333 49.9)
2,2,pt_2,2.5,POINT (-6.33667 49.9)
3,3,pt_3,3.5,POINT (-5.18 49.9)
4,4,pt_4,4.5,POINT (-4.02333 49.9)


fid            int64
name          object
value        float64
geometry    geometry
dtype: object


In [14]:
# Ensure output directory exists and write GeoParquet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / OUTPUT_FN
gdf.to_parquet(out_path, engine='pyarrow', index=False)
print('Wrote', out_path, 'size_bytes=', out_path.stat().st_size)

# Read back to verify
gdf2 = gpd.read_parquet(out_path)
print('Read back rows:', len(gdf2))
display(gdf2.head())
print('dtypes after read:')
print(gdf2.dtypes)

Wrote ../data/generated/points_100.parquet size_bytes= 10627
Read back rows: 100


,fid,name,value,geometry
0,0,pt_0,0.5,POINT (-8.65 49.9)
1,1,pt_1,1.5,POINT (-7.49333 49.9)
2,2,pt_2,2.5,POINT (-6.33667 49.9)
3,3,pt_3,3.5,POINT (-5.18 49.9)
4,4,pt_4,4.5,POINT (-4.02333 49.9)


dtypes after read:
fid            int64
name          object
value        float64
geometry    geometry
dtype: object
